### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="indian_liver_patient_dataset",
    dataset_year="2012",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5D02C",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/225/ilpd+indian+liver+patient+dataset.zip && unzip ilpd+indian+liver+patient+dataset.zip && rm ilpd+indian+liver+patient+dataset.zip
mkdir -p local-data-warehouse/indian_liver_patient_dataset && mv "Indian Liver Patient Dataset (ILPD).csv" local-data-warehouse/indian_liver_patient_dataset/
""",
    # References
    academic_reference_bibtex="""@article{ramana2012critical,
  title={A critical comparative study of liver patients from USA and INDIA: an exploratory analysis},
  author={Ramana, Bendi Venkata and Babu, M Surendra Prasad and Venkateswarlu, NB},
  journal={International Journal of Computer Science Issues (IJCSI)},
  volume={9},
  number={3},
  pages={506},
  year={2012},
  publisher={International Journal of Computer Science Issues (IJCSI)}
}
""",
    academic_reference_bibtex_key="ramana2012critical",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We keep the data as is as it does not require any further cleaning.
Note, the data contains a few (n=13) duplicates, that are likely naturally occurring duplicates given the features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Selector",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Selector",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    dataset_mold.path / "Indian Liver Patient Dataset (ILPD).csv",
    header=None,
    names=["Age", "Gender", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/G", "Selector"]
)
print("Loaded data shape:", df.shape)

as_cat_type = ["Gender", "Selector"]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()